In [1]:
import sys
import pm4py
import pandas as pd

print("Python:", sys.version)
print("PM4Py:", pm4py.__version__)
print("Pandas:", pd.__version__)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Python: 3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]
PM4Py: 2.7.23.4
Pandas: 3.0.5


In [2]:
from pathlib import Path
import pm4py

# Path to the BPI Challenge 2017 event log  
file_path = Path("../data/raw/BPI Challenge 2017.xes.gz"  )

print("File exists:", file_path.exists())
print("File:", file_path)

File exists: True
File: ..\data\raw\BPI Challenge 2017.xes.gz


In [3]:
# Load the BPI Challenge 2017 event log  

log = pm4py.read_xes(str(file_path))

print(";;Event log loaded successfully."  )
print("Number of cases:", len(log))
print("Number of events:", sum(len(trace) for trace in log))

D:\ProcessPulseBI\.venv\Lib\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

Event log loaded successfully.
Number of cases: 1202267
Number of events: 244


In [4]:
# Inspect the dataset returned by PM4Py  

print("Object type:")
print(type(log))

print("\nShape:")
print(log.shape)

print("\nColumns:")
for column in log.columns:
    print("-", column)

print("\nFirst 5 events:")
display(log.head())

Object type:
<class 'pandas.DataFrame'>

Shape:
(1202267, 19)

Columns:
- Action
- org:resource
- concept:name
- EventOrigin
- EventID
- lifecycle:transition
- time:timestamp
- case:LoanGoal
- case:ApplicationType
- case:concept:name
- case:RequestedAmount
- FirstWithdrawalAmount
- NumberOfTerms
- Accepted
- MonthlyCost
- Selected
- CreditScore
- OfferedAmount
- OfferID

First 5 events:


,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Core dataset statistics  

number_of_events = len(log)
number_of_cases = log["case:concept:name"].nunique()
number_of_activities   = log["concept:name"].nunique()
number_of_resources = log["org:resource"].nunique()

print(f"Number of events: {number_of_events:,}")
print(f"Number of cases: {number_of_cases:,}")
print(f";;Number of activities:   {number_of_activities:  ,}")
print(f";;Number of resources:   {number_of_resources:,}")

Number of events: 1,202,267
Number of cases: 31,509
Number of activities: 26
Number of resources: 149


In [6]:
# Convert timestamp to datetime  

log["time:timestamp"] = pd.to_datetime(log["time:timestamp"])

print("First event:", log["time:timestamp"].min())
print("Last event:", log["time:timestamp"].max())

First event: 2016-01-01 09:51:15.304000+00:00
Last event: 2017-02-01 14:11:03.499000+00:00


In [7]:
# Basic business dimensions  

print("Application types:")
display(log["case:ApplicationType"  ].value_counts(dropna=False))

print("\nLoan goals:")
display(log["case:LoanGoal"].value_counts(dropna=False).head(15))

Application types:


case:ApplicationType
New credit     1083767
Limit raise     118500
Name: count, dtype: int64


Loan goals:


case:LoanGoal
Car                       339798
Home improvement          294389
Existing loan takeover    227606
Other, see explanation    110643
Unknown                    85085
Remaining debt home        43874
Not speficied              41048
Extra spending limit       22964
Caravan / Camper           12967
Motorcycle                  9983
Boat                        7223
Tax payments                5557
Business goal               1090
Debt restructuring            40
Name: count, dtype: int64

In [9]:
# Most frequent process activities  

activity_counts = (
    log["concept:name"]
    .value_counts()
    .rename_axis("Activity")
    .reset_index(name="Event Count")
)

display(activity_counts)

,Activity,Event Count
0,W_Validate application,209496
1,W_Call after offers,191092
2,W_Call incomplete files,168529
3,W_Complete application,148900
4,W_Handle leads,47264
5,O_Create Offer,42995
6,O_Created,42995
7,O_Sent (mail and online),39707
8,A_Validating,38816
9,A_Create Application,31509


In [10]:
# Case duration analysis  

case_durations = (
    log.groupby("case:concept:name")["time:timestamp"]
    .agg(Case_Start="min", Case_End="max")
)

case_durations["Duration_Days"] = (
    case_durations["Case_End"] - case_durations["Case_Start"]
).dt.total_seconds() / 86_400

display(
    case_durations["Duration_Days"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95])
    .to_frame(&quot;Case Duration (Days)" )
)

,Case Duration (Days)
count,31509.000000
mean,21.899606
std,13.169233
min,0.002327
25%,11.324672
50%,19.087739
75%,31.495729
90%,35.054895
95%,42.521030
max,286.072438


In [11]:
# Duration by application type  

application_type = log.groupby("case:concept:name")["case:ApplicationType"  ].first()

duration_by_type = (
    case_durations
    .assign(Application_Type=application_type)
    .groupby("Application_Type")
    .agg(
        Cases=("Duration_Days", "size"),
        Average_Days=("Duration_Days", "mean"),
        Median_Days=("Duration_Days", "median"),
        P90_Days=("Duration_Days", lambda values: values.quantile(0.90)),
    )
    .round(2)
    .sort_values("Median_Days", ascending=False)
)

display(duration_by_type)

,Cases,Average_Days,Median_Days,P90_Days
Application_Type,,,,
New credit,28120,22.47,20.23,35.65
Limit raise,3389,17.19,13.87,31.87


In [12]:
# Duration by loan goal  

loan_goal = log.groupby("case:concept:name")["case:LoanGoal"].first()

duration_by_goal = (
    case_durations
    .assign(Loan_Goal=loan_goal)
    .groupby("Loan_Goal")
    .agg(
        Cases=("Duration_Days", "size"),
        Average_Days=("Duration_Days", "mean"),
        Median_Days=("Duration_Days", "median"),
        P90_Days=("Duration_Days", lambda values: values.quantile(0.90)),
    )
    .round(2)
    .sort_values("Median_Days", ascending=False)
)

display(duration_by_goal)

,Cases,Average_Days,Median_Days,P90_Days
Loan_Goal,,,,
Debt restructuring,2,31.18,31.18,31.73
Remaining debt home,842,29.35,29.99,51.17
Not speficied,1065,23.89,23.80,36.82
Existing loan takeover,5601,23.42,21.14,37.29
Home improvement,7669,22.44,20.01,35.52
"Other, see explanation",2985,22.17,19.79,34.99
Business goal,30,22.70,19.53,36.20
Tax payments,152,22.72,18.85,37.00
Boat,201,21.25,17.82,33.14


In [13]:
# Most common process variants (end-to-end activity paths)  

ordered_log = log.sort_values(
    ["case:concept:name", "time:timestamp"],
    kind="stable"
)

variant_counts = (
    ordered_log
    .groupby("case:concept:name", sort=False)["concept:name"]
    .agg(tuple)
    .value_counts()
)

top_variants = (
    variant_counts
    .head(5)
    .rename_axis("Variant")
    .reset_index(name="Cases")
)

top_variants["Share_of_All_Cases_%"  ] = (
    top_variants["Cases"] / number_of_cases * 100
).round(2)

top_variants["Process_Path"] = top_variants["Variant"].apply(
    lambda activities: " → ".join(activities)
)

print(f";Distinct process variants:   {len(variant_counts):,}")
display(top_variants[["Cases", "Share_of_All_Cases_%"  , "Process_Path"]])

Distinct process variants: 15,930


,Cases,Share_of_All_Cases_%,Process_Path
0,1056,3.35,A_Create Application → A_Submitted → W_Handle ...
1,1021,3.24,A_Create Application → W_Complete application ...
2,734,2.33,A_Create Application → A_Submitted → W_Handle ...
3,451,1.43,A_Create Application → A_Submitted → W_Handle ...
4,332,1.05,A_Create Application → A_Submitted → W_Handle ...


In [14]:
# Most common direct process transitions  

transitions = (
    ordered_log
    .assign(
        Next_Activity  =ordered_log
        .groupby("case:concept:name")["concept:name"]
        .shift(-1)
    )
    .dropna(subset=["Next_Activity"])
    .groupby(["concept:name", "Next_Activity"])
    .size()
    .rename("Transitions")
    .reset_index()
    .rename(columns={
        "concept:name": "From_Activity",
        "Next_Activity": "To_Activity"
    })
    .sort_values("Transitions", ascending=False)
)

transitions["Share_of_All_Transitions_%"  ] = (
    transitions["Transitions"] / transitions["Transitions"].sum() * 100
).round(2)

display(transitions.head(15))

,From_Activity,To_Activity,Transitions,Share_of_All_Transitions_%
177,W_Validate application,W_Validate application,115590,9.87
123,W_Call after offers,W_Call after offers,115569,9.87
137,W_Call incomplete files,W_Call incomplete files,113918,9.73
150,W_Complete application,W_Complete application,64695,5.53
57,O_Create Offer,O_Created,42995,3.67
167,W_Validate application,A_Validating,38816,3.32
62,O_Created,O_Sent (mail and online),35604,3.04
149,W_Complete application,W_Call after offers,31362,2.68
115,W_Call after offers,A_Complete,31362,2.68
94,O_Sent (mail and online),W_Complete application,30912,2.64


In [15]:
# Inspect event lifecycle states  

display(
    log["lifecycle:transition"  ]
    .value_counts(dropna=False)
    .rename_axis("Lifecycle State")
    .reset_index(name="Events")
)

,Lifecycle State,Events
0,complete,475306
1,suspend,215402
2,schedule,149104
3,start,128227
4,resume,127160
5,ate_abort,85224
6,withdraw,21844


In [16]:
# Keep completed events only for control-flow process mining  

completed_log = (
    log.loc[log["lifecycle:transition"  ].eq("complete")]
    .sort_values(["case:concept:name", "time:timestamp"], kind="stable")
    .copy()
)

print(f"Completed events: {len(completed_log):,}")
print(f"Cases retained: {completed_log['case:concept:name'].nunique():,}")
print(f"Activities retained:   {completed_log['concept:name'].nunique():,}")

Completed events: 475,306
Cases retained: 31,509
Activities retained: 24


In [17]:
# PM4Py process discovery: Directly-Follows Graph (DFG)  

dfg, start_activities, end_activities = pm4py.discover_dfg(completed_log)

dfg_table = (
    pd.DataFrame(
        [
            (source, target, frequency)
            for (source, target), frequency in dfg.items()
        ],
        columns=["From_Activity", "To_Activity", "Frequency"]
    )
    .sort_values("Frequency", ascending=False)
)

print("Start activities:")
display(pd.Series(start_activities).sort_values(ascending=False))

print("End activities:")
display(pd.Series(end_activities).sort_values(ascending=False))

print(&quot;Most frequent paths in the DFG:" )
display(dfg_table.head(15))

Start activities:


A_Create Application    31509
dtype: int64

End activities:


O_Cancelled                 14051
A_Pending                   10288
W_Validate application       4036
O_Refused                    1977
W_Call incomplete files       631
W_Call after offers           135
A_Cancelled                   119
W_Assess potential fraud       93
W_Complete application         63
A_Complete                     30
A_Incomplete                   29
A_Denied                       20
O_Sent (mail and online)       20
O_Sent (online only)           12
A_Validating                    3
O_Returned                      2
dtype: int64

Most frequent paths in the DFG:


,From_Activity,To_Activity,Frequency
70,O_Create Offer,O_Created,42995
77,O_Created,O_Sent (mail and online),36212
0,A_Accepted,O_Create Offer,31504
16,A_Concept,A_Accepted,31501
51,A_Validating,O_Returned,21542
20,A_Create Application,A_Submitted,20423
147,W_Complete application,A_Complete,19032
111,O_Sent (mail and online),W_Complete application,18697
9,A_Complete,A_Validating,18339
55,O_Accepted,A_Pending,17228


In [19]:
# Performance DFG: time spent between consecutive completed activities  

process_steps = completed_log.assign(
    Next_Activity=completed_log
    .groupby("case:concept:name")["concept:name"]
    .shift(-1),
    
    Next_Timestamp=completed_log
    .groupby("case:concept:name")["time:timestamp"]
    .shift(-1)
).dropna(subset=["Next_Activity", "Next_Timestamp"])

process_steps["Waiting_Hours"] = (
    process_steps["Next_Timestamp"] - process_steps["time:timestamp"]
).dt.total_seconds() / 3_600

performance_dfg = (
    process_steps.loc[process_steps["Waiting_Hours"] >= 0]
    .groupby(["concept:name", "Next_Activity"])
    .agg(
        Cases=("case:concept:name", "nunique"),
        Transitions=("Waiting_Hours", "size"),
        Median_Wait_Hours  =("Waiting_Hours", "median"),
        P90_Wait_Hours  =("Waiting_Hours", lambda values: values.quantile(0.90)),
    )
    .reset_index()
    .rename(columns={
        "concept:name": "From_Activity",
        "Next_Activity": "To_Activity",
    })
    .query("Cases >= 500")
    .round(2)
    .sort_values("Median_Wait_Hours", ascending=False)
)

display(performance_dfg.head(15))

,From_Activity,To_Activity,Cases,Transitions,Median_Wait_Hours,P90_Wait_Hours
7,A_Complete,A_Cancelled,8034,8034,736.46,741.51
98,O_Sent (mail and online),A_Cancelled,1038,1038,736.21,741.98
9,A_Complete,A_Validating,18339,18339,171.65,376.32
26,A_Incomplete,A_Cancelled,737,737,165.47,816.93
102,O_Sent (mail and online),A_Validating,3160,3235,160.36,351.09
11,A_Complete,O_Create Offer,4135,4135,98.41,503.22
30,A_Incomplete,O_Accepted,4781,4781,89.92,377.05
105,O_Sent (mail and online),O_Create Offer,571,675,74.72,497.29
88,O_Returned,A_Denied,2190,2190,52.16,144.66
117,O_Sent (online only),A_Validating,812,834,46.55,232.89


In [20]:
# Case outcomes and their duration  

case_outcomes = (
    completed_log
    .groupby("case:concept:name", sort=False)
    .agg(
        Final_Activity  =("concept:name", "last"),
        Application_Type  =("case:ApplicationType"  , "first"),
        Loan_Goal=("case:LoanGoal", "first"),
        Case_Start=("time:timestamp", "min"),
        Case_End=("time:timestamp", "max"),
    )
)

case_outcomes["Duration_Days"] = (
    case_outcomes["Case_End"] - case_outcomes["Case_Start"]
).dt.total_seconds() / 86_400

outcome_summary = (
    case_outcomes
    .groupby("Final_Activity")
    .agg(
        Cases=("Duration_Days", "size"),
        Median_Days=("Duration_Days", "median"),
        Average_Days=("Duration_Days", "mean"),
    )
)

outcome_summary["Share_of_Cases_%"] = (
    outcome_summary["Cases"] / outcome_summary["Cases"].sum() * 100
).round(2)

display(
    outcome_summary
    .round(2)
    .sort_values("Cases", ascending=False)
)

,Cases,Median_Days,Average_Days,Share_of_Cases_%
Final_Activity,,,,
O_Cancelled,14051,30.92,27.94,44.59
A_Pending,10288,13.83,16.42,32.65
W_Validate application,4036,13.75,15.83,12.81
O_Refused,1977,14.43,16.89,6.27
W_Call incomplete files,631,33.14,35.53,2.00
W_Call after offers,135,1.82,4.98,0.43
A_Cancelled,119,10.23,14.58,0.38
W_Assess potential fraud,93,21.97,25.26,0.30
W_Complete application,63,0.61,1.88,0.20


In [22]:
# Cancellation outcome by application type  

case_outcomes["Ended_Cancelled"] = (
    case_outcomes["Final_Activity"] == "O_Cancelled"
)

cancellation_by_type   = (
    case_outcomes
    .groupby("Application_Type")
    .agg(
        Cases=("Ended_Cancelled", "size"),
        Ended_Cancelled_Cases  =("Ended_Cancelled", "sum"),
        Ended_Cancelled_Pct  =(
            "Ended_Cancelled",
            lambda values: values.mean() * 100
        ),
        Median_Duration_Days  =("Duration_Days", "median"),
    )
    .round(2)
    .sort_values("Ended_Cancelled_Pct"  , ascending=False)
)

display(cancellation_by_type)

,Cases,Ended_Cancelled_Cases,Ended_Cancelled_Pct,Median_Duration_Days
Application_Type,,,,
New credit,28120,13184,46.88,20.18
Limit raise,3389,867,25.58,13.87


In [23]:
# Cancellation outcome by loan goal  
# Keep only goals with at least 100 cases for reliable comparison  

cancellation_by_goal   = (
    case_outcomes
    .groupby("Loan_Goal")
    .agg(
        Cases=("Ended_Cancelled", "size"),
        Ended_Cancelled_Cases  =("Ended_Cancelled", "sum"),
        Ended_Cancelled_Pct  =(
            "Ended_Cancelled",
            lambda values: values.mean() * 100
        ),
        Median_Duration_Days  =("Duration_Days", "median"),
    )
    .query("Cases >= 100")
    .round(2)
    .sort_values("Ended_Cancelled_Pct"  , ascending=False)
)

display(cancellation_by_goal)

,Cases,Ended_Cancelled_Cases,Ended_Cancelled_Pct,Median_Duration_Days
Loan_Goal,,,,
Not speficied,1065,564,52.96,23.79
Boat,201,105,52.24,17.82
Remaining debt home,842,433,51.43,29.66
Caravan / Camper,369,182,49.32,14.97
Car,9328,4359,46.73,16.95
"Other, see explanation",2985,1357,45.46,19.72
Existing loan takeover,5601,2499,44.62,21.10
Motorcycle,275,120,43.64,17.00
Home improvement,7669,3232,42.14,19.98


In [24]:
# Resource workload across completed process events  

resource_workload = (
    completed_log
    .dropna(subset=["org:resource"])
    .groupby("org:resource")
    .agg(
        Completed_Events  =("concept:name", "size"),
        Cases_Touched  =("case:concept:name", "nunique"),
        Unique_Activities  =("concept:name", "nunique"),
    )
    .sort_values("Completed_Events", ascending=False)
)

display(resource_workload.head(15))

,Completed_Events,Cases_Touched,Unique_Activities
org:resource,,,
User_1,75950,22473,6
User_3,9300,1804,19
User_49,9046,1662,19
User_10,8346,1705,19
User_29,8202,2824,22
User_5,7417,1868,19
User_28,7209,1551,19
User_27,6883,2584,22
User_123,6874,3238,9


In [25]:
# Activity profile of the highest-volume resource  

user_1_activity_profile   = (
    completed_log
    .loc[completed_log["org:resource"].eq("User_1")]
    .groupby("concept:name")
    .size()
    .rename("Completed_Events")
    .sort_values(ascending=False)
    .to_frame()
)

user_1_activity_profile  ["Share_of_User_1_Events_Pct"  ] = (
    user_1_activity_profile  ["Completed_Events"]
    / user_1_activity_profile  ["Completed_Events"].sum()
    * 100
).round(2)

display(user_1_activity_profile  )

,Completed_Events,Share_of_User_1_Events_Pct
concept:name,,
A_Submitted,20423,26.89
A_Create Application,20423,26.89
A_Concept,16997,22.38
O_Cancelled,9982,13.14
A_Cancelled,7953,10.47
W_Handle leads,172,0.23


In [27]:
from getpass import getpass

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

db_url = URL.create(
    "postgresql+psycopg2"  ,
    username="postgres",
    password=getpass("PostgreSQL password: &;quot; ), 
    host="localhost",
    port=5432,
    database="processpulse_bi",
)

engine = create_engine(db_url, pool_pre_ping=True)

with engine.connect() as connection:
    database_name, database_user = connection.execute(
        text(";SELECT current_database(), current_user"  )
    ).one()

print(f"Connected to database:   {database_name}")
print(f"Connected as user:   {database_user}")

PostgreSQL password:  ········


Connected to database: processpulse_bi
Connected as user: postgres


In [29]:
# Prepare a database-friendly event table  

column_mapping = {
    "Action": "action",
    "org:resource": "resource",
    "concept:name": "activity",
    "EventOrigin": "event_origin",
    "EventID": "event_id",
    "lifecycle:transition"  : "lifecycle_transition"  ,
    "time:timestamp": "event_timestamp",
    "case:LoanGoal": "loan_goal",
    "case:ApplicationType"  : "application_type",
    "case:concept:name": "case_id",
    "case:RequestedAmount"  : "requested_amount",
    "FirstWithdrawalAmount"  : "first_withdrawal_amount"  ,
    "NumberOfTerms": "number_of_terms",
    "Accepted": "accepted",
    "MonthlyCost": "monthly_cost",
    "Selected": "selected",
    "CreditScore": "credit_score",
    "OfferedAmount": "offered_amount",
    "OfferID": "offer_id",
}

events_for_db = log.rename(columns=column_mapping).copy()
events_for_db["event_timestamp"] = pd.to_datetime(
    events_for_db["event_timestamp"]
)

print(f"Rows prepared: {len(events_for_db):,}")
print(f"Columns prepared: {len(events_for_db.columns)}")

display(events_for_db.head(3))

Rows prepared: 1,202,267
Columns prepared: 19


,action,resource,activity,event_origin,event_id,lifecycle_transition,event_timestamp,loan_goal,application_type,case_id,requested_amount,first_withdrawal_amount,number_of_terms,accepted,monthly_cost,selected,credit_score,offered_amount,offer_id
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
from sqlalchemy import types

print("Uploading raw events to PostgreSQL..."  )

events_for_db.to_sql(
    name="bpi_2017_events",
    con=engine,
    schema="raw",
    if_exists="replace",
    index=False,
    chunksize=1_000,
    method="multi",
    dtype={
        "event_timestamp": types.DateTime(timezone=True),
    },
)

with engine.begin() as connection:
    connection.execute(
        text(
            ";CREATE INDEX idx_raw_events_case_id &;quot;  
            &quot;ON raw.bpi_2017_events (case_id)" 
        )
    )
    connection.execute(
        text(
            ";CREATE INDEX idx_raw_events_timestamp &;quot;  
            &quot;ON raw.bpi_2017_events (event_timestamp)" 
        )
    )

print("Upload complete.")

Uploading raw events to PostgreSQL...
Upload complete.


In [31]:
with engine.connect() as connection:
    database_event_count   = connection.execute(
        text(";SELECT COUNT(*) FROM raw.bpi_2017_events"  )
    ).scalar_one()

print(f";;Events stored in PostgreSQL:   {database_event_count:  ,}")

Events stored in PostgreSQL: 1,202,267


In [32]:
with engine.begin() as connection:
    connection.execute(
        text(&quot;DROP TABLE IF EXISTS analytics .case_summary" )
    )

    connection.execute(
        text(
            """
            CREATE TABLE analytics.case_summary AS  
            WITH case_metrics AS (  
                SELECT  
                    case_id,  
                    MIN(event_timestamp) AS case_start,  
                    MAX(event_timestamp) AS case_end,  
                    COUNT(*) AS event_count,  
                    COUNT(DISTINCT activity) AS unique_activity_count  
                FROM raw.bpi_2017_events  
                GROUP BY case_id  
            ),
            case_attributes AS (  
                SELECT DISTINCT ON (case_id)  
                    case_id,  
                    application_type,  
                    loan_goal,  
                    requested_amount,  
                    credit_score  
                FROM raw.bpi_2017_events  
                ORDER BY case_id, event_timestamp  
            ),
            final_completed_event AS (  
                SELECT DISTINCT ON (case_id)  
                    case_id,  
                    activity AS final_activity  
                FROM raw.bpi_2017_events  
                WHERE lifecycle_transition = 'complete'  
                ORDER BY case_id, event_timestamp DESC, event_id DESC  
            )
            SELECT
                metrics.case_id,  
                attributes.application_type,  
                attributes.loan_goal,  
                attributes.requested_amount,  
                attributes.credit_score,  
                metrics.case_start,  
                metrics.case_end,  
                EXTRACT(  
                    EPOCH FROM (metrics.case_end - metrics.case_start)  
                ) / 86400.0 AS duration_days,  
                metrics.event_count,  
                metrics.unique_activity_count,  
                final_event.final_activity,  
                COALESCE(  
                    final_event.final_activity = 'O_Cancelled',  
                    FALSE  
                ) AS ended_cancelled  
            FROM case_metrics AS metrics  
            LEFT JOIN case_attributes AS attributes  
                ON metrics.case_id = attributes.case_id  
            LEFT JOIN final_completed_event AS final_event  
                ON metrics.case_id = final_event.case_id  
            """
        )
    )

    connection.execute(
        text(
            ";CREATE INDEX idx_case_summary_case_id &;quot;  
            &quot;ON analytics.case_summary (case_id)" 
        )
    )

print("analytics.case_summary created."  )

analytics.case_summary created.


In [33]:
with engine.begin() as connection:
    connection.execute(
        text(&quot;DROP TABLE IF EXISTS analytics .case_summary" )
    )

    connection.execute(
        text(
            """
            CREATE TABLE analytics.case_summary AS  
            WITH case_metrics AS (  
                SELECT  
                    case_id,  
                    MIN(event_timestamp) AS case_start,  
                    MAX(event_timestamp) AS case_end,  
                    COUNT(*) AS event_count,  
                    COUNT(DISTINCT activity) AS unique_activity_count  
                FROM raw.bpi_2017_events  
                GROUP BY case_id  
            ),
            case_attributes AS (  
                SELECT DISTINCT ON (case_id)  
                    case_id,  
                    application_type,  
                    loan_goal,  
                    requested_amount,  
                    credit_score  
                FROM raw.bpi_2017_events  
                ORDER BY case_id, event_timestamp  
            ),
            final_completed_event AS (  
                SELECT DISTINCT ON (case_id)  
                    case_id,  
                    activity AS final_activity  
                FROM raw.bpi_2017_events  
                WHERE lifecycle_transition = 'complete'  
                ORDER BY case_id, event_timestamp DESC, event_id DESC  
            )
            SELECT
                metrics.case_id,  
                attributes.application_type,  
                attributes.loan_goal,  
                attributes.requested_amount,  
                attributes.credit_score,  
                metrics.case_start,  
                metrics.case_end,  
                EXTRACT(  
                    EPOCH FROM (metrics.case_end - metrics.case_start)  
                ) / 86400.0 AS duration_days,  
                metrics.event_count,  
                metrics.unique_activity_count,  
                final_event.final_activity,  
                COALESCE(  
                    final_event.final_activity = 'O_Cancelled',  
                    FALSE  
                ) AS ended_cancelled  
            FROM case_metrics AS metrics  
            LEFT JOIN case_attributes AS attributes  
                ON metrics.case_id = attributes.case_id  
            LEFT JOIN final_completed_event AS final_event  
                ON metrics.case_id = final_event.case_id  
            """
        )
    )

    connection.execute(
        text(
            ";CREATE INDEX idx_case_summary_case_id &;quot;  
            &quot;ON analytics.case_summary (case_id)" 
        )
    )

print("analytics.case_summary created."  )

analytics.case_summary created.


In [34]:
database_kpis = pd.read_sql(
    text(
        """
        SELECT
            COUNT(*) AS total_cases,  
            ROUND(AVG(duration_days)::numeric, 2) AS average_duration_days,  
            ROUND(
                100.0 * AVG(  
                    CASE WHEN ended_cancelled THEN 1.0 ELSE 0.0 END  
                ),
                2
            ) AS ended_cancelled_pct  
        FROM analytics.case_summary  
        """
    ),
    engine,
)

display(database_kpis)

,total_cases,average_duration_days,ended_cancelled_pct
0,31509,21.9,44.59


In [42]:
import os
import pm4py

# 1. Output directory
output_dir = "exports_page_2"
os.makedirs(output_dir, exist_ok=True)

# 2. Filter completed log to the Top 10 most frequent process variants (Happy Paths)
filtered_log = pm4py.filter_variants_top_k(completed_log, 10)

print(f"Original completed cases: {completed_log['case:concept:name'].nunique():,}")
print(f"Clean filtered cases (Top 10 variants): {filtered_log['case:concept:name'].nunique():,}")

# ---------------------------------------------------------
# A. Clean Frequency DFG (Happy Path Flow)
# ---------------------------------------------------------
clean_dfg, clean_start, clean_end = pm4py.discover_dfg(filtered_log)

pm4py.save_vis_dfg(
    clean_dfg,
    clean_start,
    clean_end,
    file_path=os.path.join(output_dir, "dfg_frequency_clean.png")
)
print("Saved clean frequency graph: dfg_frequency_clean.png")

# ---------------------------------------------------------
# B. Clean Performance DFG (Happy Path Bottlenecks)
# ---------------------------------------------------------
perf_clean_dfg, perf_clean_start, perf_clean_end = pm4py.discover_performance_dfg(filtered_log)

pm4py.save_vis_performance_dfg(
    perf_clean_dfg,
    perf_clean_start,
    perf_clean_end,
    file_path=os.path.join(output_dir, "dfg_performance_clean.png")
)
print("Saved clean performance graph: dfg_performance_clean.png")

Original completed cases: 31,509
Clean filtered cases (Top 10 variants): 9,817
Saved clean frequency graph: dfg_frequency_clean.png
Saved clean performance graph: dfg_performance_clean.png
